# Data Analyst Agent: CSV/XLSX Question-Answering with Tool Calling

**Goal:** build an agent that loads tabular data files, answers natural-language questions about the data, computes statistics, and suggests appropriate charts — all driven by a tool-calling architecture.

---

## The Problem

Business users have spreadsheets. They have questions. They don't know pandas.

An analyst agent bridges that gap: the user asks a plain-English question, and the agent figures out which operations to run, executes them, and returns a clear answer with a chart recommendation.

## What We Build

```text
User: "What's the average revenue by region, and which region is growing fastest?"
                    │
                    ▼
            ┌───────────────┐
            │  Agent Router  │  Understands intent, plans tool calls
            └───────┬───────┘
                    │
        ┌───────────┼───────────┐
        ▼           ▼           ▼
  ┌──────────┐ ┌──────────┐ ┌──────────────┐
  │load_file │ │compute_  │ │suggest_chart │
  │          │ │statistics│ │              │
  └──────────┘ └──────────┘ └──────────────┘
        │           │           │
        └───────────┼───────────┘
                    ▼
            ┌───────────────┐
            │  Final Answer  │  Text + stats + chart recommendation
            └───────────────┘
```

## Key Concepts

- **Tool-calling design**: how to define, register, and dispatch tools
- **Schema-driven tools**: typed parameters with validation
- **Multi-step reasoning**: agent chains tool calls to answer complex questions
- **Chart suggestion logic**: matching data shape to appropriate visualization


## 1. Environment Setup

In [ ]:
!pip install -q pandas numpy openpyxl matplotlib seaborn

In [ ]:
import io
import json
import random
import re
import textwrap
from collections import defaultdict
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path.cwd()
ARTIFACT_DIR = PROJECT_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

print(f"Project dir: {PROJECT_DIR}")

## 2. Tool-Calling Design — Concepts

### What Is Tool Calling?

Tool calling is a pattern where an LLM does not answer directly. Instead, it emits a structured request to invoke a function, receives the result, and then uses that result to formulate its answer.

```text
User question ──► LLM ──► Tool call request ──► Framework executes tool
                                                        │
                   LLM ◄── Tool result ◄────────────────┘
                    │
                    ▼
              Final answer
```

### Why Tool Calling?

| Approach | Strength | Weakness |
|---|---|---|
| LLM generates code | Flexible, can do anything | Unsafe, hallucinated code, hard to validate |
| LLM calls pre-built tools | Safe, predictable, testable | Limited to defined tools |
| Hybrid (tools + code sandbox) | Best of both | Most complex to build |

We use the **pre-built tools** approach: every operation is a well-defined function with typed inputs and outputs.

### Tool Definition Anatomy

Every tool needs four things:

```python
{
    "name": "compute_statistics",        # Unique name
    "description": "Compute descriptive  # What it does (LLM reads this)
                    statistics for a
                    column",
    "parameters": {                      # Typed inputs
        "column": {"type": "string", "description": "Column name"},
        "stats": {"type": "array", "items": "string", "description": "Which stats"},
    },
    "returns": "dict"                    # What comes back
}
```

### The Tool Registry Pattern

Instead of hard-coding tool dispatch, we use a registry:

```python
registry = {}

def register_tool(name, func, schema):
    registry[name] = {"func": func, "schema": schema}

def dispatch(name, args):
    tool = registry[name]
    validate(args, tool["schema"])  # Safety check
    return tool["func"](**args)
```

This pattern is used by OpenAI, Anthropic, LangChain, and most agent frameworks.


## 3. Synthetic Dataset

We create a realistic sales dataset to demonstrate the agent. In production, the agent would load user-uploaded files.


In [ ]:
# Generate a synthetic sales dataset
np.random.seed(SEED)

n_rows = 500
regions = ["North", "South", "East", "West"]
products = ["Widget A", "Widget B", "Gadget X", "Gadget Y", "Service Z"]
channels = ["Online", "Retail", "Wholesale"]

dates = pd.date_range("2023-01-01", "2024-12-31", periods=n_rows)

data = {
    "order_id": [f"ORD-{i:04d}" for i in range(1, n_rows + 1)],
    "date": np.sort(dates),
    "region": np.random.choice(regions, n_rows, p=[0.3, 0.25, 0.25, 0.2]),
    "product": np.random.choice(products, n_rows, p=[0.25, 0.25, 0.2, 0.15, 0.15]),
    "channel": np.random.choice(channels, n_rows, p=[0.45, 0.35, 0.20]),
    "quantity": np.random.randint(1, 50, n_rows),
    "unit_price": np.round(np.random.uniform(10, 500, n_rows), 2),
    "discount_pct": np.random.choice([0, 5, 10, 15, 20], n_rows, p=[0.4, 0.25, 0.2, 0.1, 0.05]),
    "customer_rating": np.round(np.clip(np.random.normal(4.0, 0.8, n_rows), 1, 5), 1),
}

df = pd.DataFrame(data)
df["revenue"] = np.round(df["quantity"] * df["unit_price"] * (1 - df["discount_pct"] / 100), 2)
df["month"] = df["date"].dt.to_period("M").astype(str)

# Save as CSV and XLSX
csv_path = ARTIFACT_DIR / "sales_data.csv"
xlsx_path = ARTIFACT_DIR / "sales_data.xlsx"
df.to_csv(csv_path, index=False)
df.to_excel(xlsx_path, index=False)

print(f"Dataset: {len(df)} rows x {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
print(f"Saved: {csv_path.name}, {xlsx_path.name}")
print(f"\nSample:")
df.head(3)

In [ ]:
print("Data types:")
print(df.dtypes.to_string())
print(f"\nNumeric columns: {list(df.select_dtypes(include='number').columns)}")
print(f"Categorical columns: {list(df.select_dtypes(include='object').columns)}")
print(f"Date columns: {[c for c in df.columns if pd.api.types.is_datetime64_any_dtype(df[c])]}")

## 4. The Tool Registry

We build a typed tool registry that:
1. Stores tool functions with their schemas
2. Validates inputs before execution
3. Logs every call for observability
4. Returns structured results


In [ ]:
@dataclass
class ToolParameter:
    """Schema for a single tool parameter."""
    name: str
    type: str  # "string", "number", "boolean", "array"
    description: str
    required: bool = True
    enum: Optional[list] = None  # Allowed values
    default: Any = None


@dataclass
class ToolSchema:
    """Schema for a tool."""
    name: str
    description: str
    parameters: list[ToolParameter]
    returns: str  # Description of return value


@dataclass
class ToolResult:
    """Result of a tool execution."""
    tool_name: str
    success: bool
    data: Any
    error: Optional[str] = None
    execution_time_ms: float = 0


class ToolRegistry:
    """Registry for agent tools with validation and logging."""

    def __init__(self):
        self._tools: dict[str, dict] = {}
        self._call_log: list[dict] = []

    def register(self, schema: ToolSchema, func: Callable):
        """Register a tool with its schema and implementation."""
        self._tools[schema.name] = {"schema": schema, "func": func}

    def list_tools(self) -> list[dict]:
        """List all registered tools (for the LLM system prompt)."""
        tools = []
        for name, tool in self._tools.items():
            s = tool["schema"]
            tools.append({
                "name": s.name,
                "description": s.description,
                "parameters": [
                    {"name": p.name, "type": p.type, "description": p.description,
                     "required": p.required, "enum": p.enum}
                    for p in s.parameters
                ],
                "returns": s.returns,
            })
        return tools

    def validate_args(self, tool_name: str, args: dict) -> list[str]:
        """Validate arguments against the tool schema."""
        errors = []
        schema = self._tools[tool_name]["schema"]
        param_map = {p.name: p for p in schema.parameters}

        # Check required params
        for p in schema.parameters:
            if p.required and p.name not in args:
                errors.append(f"Missing required parameter: {p.name}")

        # Check enum constraints
        for key, val in args.items():
            if key in param_map and param_map[key].enum:
                if val not in param_map[key].enum:
                    errors.append(f"{key}={val} not in allowed values: {param_map[key].enum}")

        return errors

    def call(self, tool_name: str, args: dict) -> ToolResult:
        """Execute a tool with validation and logging."""
        if tool_name not in self._tools:
            return ToolResult(tool_name=tool_name, success=False,
                              data=None, error=f"Unknown tool: {tool_name}")

        # Validate
        errors = self.validate_args(tool_name, args)
        if errors:
            return ToolResult(tool_name=tool_name, success=False,
                              data=None, error="; ".join(errors))

        # Execute
        import time
        start = time.perf_counter()
        try:
            result = self._tools[tool_name]["func"](**args)
            elapsed = (time.perf_counter() - start) * 1000
            tool_result = ToolResult(tool_name=tool_name, success=True,
                                     data=result, execution_time_ms=round(elapsed, 2))
        except Exception as e:
            elapsed = (time.perf_counter() - start) * 1000
            tool_result = ToolResult(tool_name=tool_name, success=False,
                                     data=None, error=str(e),
                                     execution_time_ms=round(elapsed, 2))

        # Log
        self._call_log.append({
            "tool": tool_name,
            "args": {k: str(v)[:100] for k, v in args.items()},
            "success": tool_result.success,
            "time_ms": tool_result.execution_time_ms,
            "error": tool_result.error,
        })

        return tool_result

    @property
    def call_log(self):
        return self._call_log


registry = ToolRegistry()
print(f"ToolRegistry created. Methods: register, list_tools, validate_args, call")

## 5. Implementing the Agent's Tools

We define six tools:

| Tool | Purpose | Key parameters |
|---|---|---|
| `load_file` | Load a CSV or XLSX file | file_path |
| `describe_data` | Get column names, types, row count, sample | (none — uses loaded data) |
| `compute_statistics` | Compute stats for a column | column, stats list |
| `group_aggregate` | Group by one column, aggregate another | group_col, value_col, agg_func |
| `filter_data` | Filter rows by condition | column, operator, value |
| `suggest_chart` | Recommend a chart type and config | question, columns |


In [ ]:
# Shared state: the loaded DataFrame
_agent_state = {"df": None, "file_path": None, "file_name": None}


# ── Tool 1: load_file ──
def load_file(file_path: str) -> dict:
    """Load a CSV or XLSX file into the agent's working memory."""
    p = Path(file_path)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    suffix = p.suffix.lower()
    if suffix == ".csv":
        loaded = pd.read_csv(p)
    elif suffix in (".xlsx", ".xls"):
        loaded = pd.read_excel(p)
    else:
        raise ValueError(f"Unsupported format: {suffix}. Use .csv or .xlsx")

    _agent_state["df"] = loaded
    _agent_state["file_path"] = str(p)
    _agent_state["file_name"] = p.name

    return {
        "file": p.name,
        "rows": len(loaded),
        "columns": list(loaded.columns),
        "dtypes": {col: str(dtype) for col, dtype in loaded.dtypes.items()},
    }


registry.register(
    ToolSchema(
        name="load_file",
        description="Load a CSV or XLSX file. This must be called before any analysis.",
        parameters=[
            ToolParameter("file_path", "string", "Path to the .csv or .xlsx file"),
        ],
        returns="Dict with file name, row count, column names, and data types",
    ),
    load_file,
)


# ── Tool 2: describe_data ──
def describe_data() -> dict:
    """Get an overview of the currently loaded data."""
    loaded = _agent_state["df"]
    if loaded is None:
        raise RuntimeError("No file loaded. Call load_file first.")

    numeric_cols = list(loaded.select_dtypes(include="number").columns)
    categorical_cols = list(loaded.select_dtypes(include=["object", "category"]).columns)
    date_cols = [c for c in loaded.columns if pd.api.types.is_datetime64_any_dtype(loaded[c])]

    overview = {
        "file": _agent_state["file_name"],
        "shape": {"rows": len(loaded), "columns": len(loaded.columns)},
        "columns": {
            "numeric": numeric_cols,
            "categorical": categorical_cols,
            "datetime": date_cols,
            "all": list(loaded.columns),
        },
        "missing_values": loaded.isnull().sum().to_dict(),
        "sample_rows": loaded.head(3).to_dict(orient="records"),
    }
    return overview


registry.register(
    ToolSchema(
        name="describe_data",
        description="Get column names, types, missing values, and sample rows of the loaded data.",
        parameters=[],
        returns="Dict with shape, column types, missing values, and 3 sample rows",
    ),
    describe_data,
)


# ── Tool 3: compute_statistics ──
def compute_statistics(column: str, stats: Optional[list] = None) -> dict:
    """Compute descriptive statistics for a numeric column."""
    loaded = _agent_state["df"]
    if loaded is None:
        raise RuntimeError("No file loaded.")
    if column not in loaded.columns:
        raise ValueError(f"Column '{column}' not found. Available: {list(loaded.columns)}")

    col = loaded[column]
    if not pd.api.types.is_numeric_dtype(col):
        # For non-numeric: value counts
        vc = col.value_counts()
        return {
            "column": column,
            "type": "categorical",
            "unique_values": int(col.nunique()),
            "top_values": vc.head(10).to_dict(),
            "missing": int(col.isnull().sum()),
        }

    available = {
        "mean": col.mean, "median": col.median, "std": col.std,
        "min": col.min, "max": col.max, "sum": col.sum,
        "count": col.count, "q25": lambda: col.quantile(0.25),
        "q75": lambda: col.quantile(0.75),
    }

    if stats is None:
        stats = ["mean", "median", "std", "min", "max", "count"]

    result = {"column": column, "type": "numeric"}
    for s in stats:
        if s in available:
            val = available[s]()
            result[s] = round(float(val), 4) if pd.notna(val) else None
    result["missing"] = int(col.isnull().sum())
    return result


registry.register(
    ToolSchema(
        name="compute_statistics",
        description="Compute descriptive statistics (mean, median, std, min, max, etc.) for a column.",
        parameters=[
            ToolParameter("column", "string", "Column name to analyze"),
            ToolParameter("stats", "array", "List of stats: mean, median, std, min, max, sum, count, q25, q75",
                          required=False, default=None),
        ],
        returns="Dict with computed statistics for the column",
    ),
    compute_statistics,
)


# ── Tool 4: group_aggregate ──
def group_aggregate(group_col: str, value_col: str,
                    agg_func: str = "mean", top_n: Optional[int] = None) -> dict:
    """Group by one column and aggregate another."""
    loaded = _agent_state["df"]
    if loaded is None:
        raise RuntimeError("No file loaded.")

    for col_name in [group_col, value_col]:
        if col_name not in loaded.columns:
            raise ValueError(f"Column '{col_name}' not found. Available: {list(loaded.columns)}")

    allowed_aggs = {"mean", "median", "sum", "count", "min", "max", "std"}
    if agg_func not in allowed_aggs:
        raise ValueError(f"agg_func '{agg_func}' not supported. Use: {allowed_aggs}")

    grouped = loaded.groupby(group_col)[value_col].agg(agg_func)
    if top_n:
        grouped = grouped.nlargest(top_n)

    return {
        "group_col": group_col,
        "value_col": value_col,
        "agg_func": agg_func,
        "results": {str(k): round(float(v), 4) if pd.notna(v) else None
                     for k, v in grouped.items()},
        "num_groups": len(grouped),
    }


registry.register(
    ToolSchema(
        name="group_aggregate",
        description="Group the data by one column and compute an aggregate (mean, sum, etc.) of another column.",
        parameters=[
            ToolParameter("group_col", "string", "Column to group by"),
            ToolParameter("value_col", "string", "Column to aggregate"),
            ToolParameter("agg_func", "string", "Aggregation function",
                          required=False, enum=["mean", "median", "sum", "count", "min", "max", "std"]),
            ToolParameter("top_n", "number", "Return only top N groups by value",
                          required=False, default=None),
        ],
        returns="Dict with grouped results",
    ),
    group_aggregate,
)


# ── Tool 5: filter_data ──
def filter_data(column: str, operator: str, value: Any,
                return_stats: bool = True) -> dict:
    """Filter rows by a condition and return summary or matching rows."""
    loaded = _agent_state["df"]
    if loaded is None:
        raise RuntimeError("No file loaded.")
    if column not in loaded.columns:
        raise ValueError(f"Column '{column}' not found.")

    ops = {
        "==": lambda s, v: s == v,
        "!=": lambda s, v: s != v,
        ">": lambda s, v: s > v,
        ">=": lambda s, v: s >= v,
        "<": lambda s, v: s < v,
        "<=": lambda s, v: s <= v,
        "contains": lambda s, v: s.astype(str).str.contains(str(v), case=False, na=False),
    }

    if operator not in ops:
        raise ValueError(f"Operator '{operator}' not supported. Use: {list(ops.keys())}")

    mask = ops[operator](loaded[column], value)
    filtered = loaded[mask]

    result = {
        "column": column,
        "operator": operator,
        "value": value,
        "total_rows": len(loaded),
        "matching_rows": len(filtered),
        "match_pct": round(len(filtered) / len(loaded) * 100, 2),
    }

    if return_stats and len(filtered) > 0:
        numeric_cols = filtered.select_dtypes(include="number").columns
        if len(numeric_cols) > 0:
            result["numeric_summary"] = {}
            for col in numeric_cols[:5]:  # Limit to 5 columns
                result["numeric_summary"][col] = {
                    "mean": round(float(filtered[col].mean()), 4),
                    "sum": round(float(filtered[col].sum()), 4),
                }

    return result


registry.register(
    ToolSchema(
        name="filter_data",
        description="Filter rows where a column meets a condition (==, !=, >, <, >=, <=, contains). Returns count and stats of matching rows.",
        parameters=[
            ToolParameter("column", "string", "Column to filter on"),
            ToolParameter("operator", "string", "Comparison operator",
                          enum=["==", "!=", ">", "<", ">=", "<=", "contains"]),
            ToolParameter("value", "string", "Value to compare against"),
            ToolParameter("return_stats", "boolean", "Include summary stats of matching rows",
                          required=False, default=True),
        ],
        returns="Dict with match count, percentage, and optional numeric summary",
    ),
    filter_data,
)


# ── Tool 6: suggest_chart ──
def suggest_chart(question: str, x_column: Optional[str] = None,
                  y_column: Optional[str] = None,
                  group_column: Optional[str] = None) -> dict:
    """Suggest the best chart type for visualizing the answer to a question."""
    loaded = _agent_state["df"]
    if loaded is None:
        raise RuntimeError("No file loaded.")

    suggestions = []

    # Determine column types
    def col_type(col_name):
        if col_name and col_name in loaded.columns:
            if pd.api.types.is_numeric_dtype(loaded[col_name]):
                return "numeric"
            elif pd.api.types.is_datetime64_any_dtype(loaded[col_name]):
                return "datetime"
            else:
                nunique = loaded[col_name].nunique()
                return "categorical_few" if nunique <= 10 else "categorical_many"
        return None

    x_type = col_type(x_column)
    y_type = col_type(y_column)
    q_lower = question.lower()

    # Rule-based chart suggestion
    if any(w in q_lower for w in ["trend", "over time", "time series", "monthly", "growth"]):
        suggestions.append({
            "chart_type": "line",
            "reason": "Time-based question suggests a line chart to show trends",
            "x": x_column or "date",
            "y": y_column,
            "config": {"marker": True, "sort_x": True},
        })

    if any(w in q_lower for w in ["compare", "by region", "by product", "by channel", "per"]):
        suggestions.append({
            "chart_type": "bar",
            "reason": "Comparison across categories is best shown with a bar chart",
            "x": x_column,
            "y": y_column,
            "config": {"horizontal": False, "sort_values": True},
        })

    if any(w in q_lower for w in ["distribution", "spread", "range", "histogram"]):
        suggestions.append({
            "chart_type": "histogram",
            "reason": "Distribution questions are best answered with histograms",
            "x": x_column or y_column,
            "y": None,
            "config": {"bins": 20, "kde": True},
        })

    if any(w in q_lower for w in ["relationship", "correlation", "vs", "versus"]):
        suggestions.append({
            "chart_type": "scatter",
            "reason": "Relationship between two variables shown via scatter plot",
            "x": x_column,
            "y": y_column,
            "config": {"alpha": 0.6, "hue": group_column},
        })

    if any(w in q_lower for w in ["proportion", "share", "percentage", "breakdown"]):
        suggestions.append({
            "chart_type": "pie",
            "reason": "Proportional breakdown shown with pie/donut chart",
            "x": x_column,
            "y": y_column,
            "config": {"donut": True, "max_slices": 8},
        })
        suggestions.append({
            "chart_type": "stacked_bar",
            "reason": "Alternative: stacked bar chart is often clearer than pie for multiple categories",
            "x": x_column,
            "y": y_column,
            "config": {"normalize": True},
        })

    if any(w in q_lower for w in ["top", "best", "worst", "highest", "lowest", "ranking"]):
        suggestions.append({
            "chart_type": "horizontal_bar",
            "reason": "Rankings are clearest as horizontal bar charts sorted by value",
            "x": y_column,
            "y": x_column,
            "config": {"sort_values": True, "top_n": 10},
        })

    # Fallback based on column types
    if not suggestions:
        if x_type == "categorical_few" and y_type == "numeric":
            suggestions.append({
                "chart_type": "bar",
                "reason": "Categorical x + numeric y defaults to bar chart",
                "x": x_column, "y": y_column, "config": {},
            })
        elif x_type == "numeric" and y_type == "numeric":
            suggestions.append({
                "chart_type": "scatter",
                "reason": "Two numeric columns defaults to scatter plot",
                "x": x_column, "y": y_column, "config": {"alpha": 0.6},
            })
        else:
            suggestions.append({
                "chart_type": "bar",
                "reason": "Default recommendation for general questions",
                "x": x_column, "y": y_column, "config": {},
            })

    return {
        "question": question,
        "suggestions": suggestions,
        "primary": suggestions[0] if suggestions else None,
    }


registry.register(
    ToolSchema(
        name="suggest_chart",
        description="Suggest the best chart type for visualizing an answer. Analyzes the question and column types to recommend bar, line, scatter, histogram, or pie charts.",
        parameters=[
            ToolParameter("question", "string", "The user's question (used to infer chart type)"),
            ToolParameter("x_column", "string", "Column for the x-axis", required=False),
            ToolParameter("y_column", "string", "Column for the y-axis", required=False),
            ToolParameter("group_column", "string", "Column for grouping/color", required=False),
        ],
        returns="Dict with chart type suggestions, reasons, and configuration",
    ),
    suggest_chart,
)


print(f"Registered {len(registry._tools)} tools:")
for t in registry.list_tools():
    params = ", ".join(p["name"] for p in t["parameters"])
    print(f"  {t['name']}({params})")

## 6. Tool Schemas for the LLM System Prompt

In production, these schemas are included in the LLM's system prompt so it knows what tools are available and how to call them.


In [ ]:
# Generate the tool schema that would go in an LLM system prompt
tool_schemas = registry.list_tools()

system_prompt_tools = "You have the following tools available:\n\n"
for tool in tool_schemas:
    system_prompt_tools += f"### {tool['name']}\n"
    system_prompt_tools += f"{tool['description']}\n"
    system_prompt_tools += "Parameters:\n"
    for p in tool["parameters"]:
        req = "required" if p["required"] else "optional"
        enum_str = f" (values: {p['enum']})" if p.get("enum") else ""
        system_prompt_tools += f"  - {p['name']} ({p['type']}, {req}): {p['description']}{enum_str}\n"
    system_prompt_tools += f"Returns: {tool['returns']}\n\n"

print(system_prompt_tools)

## 7. The Data Analyst Agent

The agent receives a natural-language question, plans which tools to call, executes them in sequence, and assembles a final answer.

In production, an LLM does the planning. Here we implement rule-based intent detection to show the tool-calling mechanics clearly without requiring API keys.


In [ ]:
class DataAnalystAgent:
    """An agent that answers questions about tabular data using tools."""

    def __init__(self, tool_registry: ToolRegistry):
        self.registry = tool_registry
        self.conversation: list[dict] = []

    def _detect_intent(self, question: str) -> list[dict]:
        """Parse the user's question into a sequence of tool calls.

        In production, an LLM does this. Here we use keyword matching.
        """
        q = question.lower()
        plan = []

        # Always ensure data is loaded
        if _agent_state["df"] is None:
            return [{"error": "No data loaded. Please load a file first."}]

        loaded = _agent_state["df"]
        numeric_cols = list(loaded.select_dtypes(include="number").columns)
        categorical_cols = list(loaded.select_dtypes(include=["object", "category"]).columns)

        # Detect column references in the question
        mentioned_cols = [c for c in loaded.columns if c.lower() in q.replace("_", " ")]

        # Intent: overview / describe
        if any(w in q for w in ["overview", "describe", "what columns", "what data",
                                 "tell me about", "what's in"]):
            plan.append({"tool": "describe_data", "args": {}})
            return plan

        # Intent: statistics for a column
        if any(w in q for w in ["average", "mean", "median", "sum", "total",
                                 "standard deviation", "min", "max", "statistics"]):
            target_col = None
            for c in numeric_cols:
                if c.lower() in q.replace("_", " "):
                    target_col = c
                    break
            if not target_col:
                target_col = "revenue" if "revenue" in numeric_cols else numeric_cols[0]

            stats = []
            if "average" in q or "mean" in q:
                stats.append("mean")
            if "median" in q:
                stats.append("median")
            if "sum" in q or "total" in q:
                stats.append("sum")
            if "std" in q or "deviation" in q:
                stats.append("std")
            if not stats:
                stats = ["mean", "median", "std", "min", "max"]

            plan.append({"tool": "compute_statistics",
                         "args": {"column": target_col, "stats": stats}})

        # Intent: group by / compare
        if any(w in q for w in ["by region", "by product", "by channel", "per region",
                                 "per product", "per channel", "each region", "each product",
                                 "compare", "breakdown"]):
            group_col = None
            for candidate in ["region", "product", "channel"]:
                if candidate in q:
                    group_col = candidate
                    break
            if not group_col:
                group_col = categorical_cols[0] if categorical_cols else None

            value_col = None
            for candidate in ["revenue", "quantity", "unit_price", "customer_rating", "discount_pct"]:
                if candidate.replace("_", " ") in q.replace("_", " "):
                    value_col = candidate
                    break
            if not value_col:
                value_col = "revenue"

            agg = "mean"
            if "total" in q or "sum" in q:
                agg = "sum"
            elif "median" in q:
                agg = "median"
            elif "count" in q or "how many" in q:
                agg = "count"
            elif "max" in q or "highest" in q or "best" in q:
                agg = "max"

            if group_col:
                plan.append({"tool": "group_aggregate",
                             "args": {"group_col": group_col, "value_col": value_col,
                                      "agg_func": agg}})

        # Intent: filter
        if any(w in q for w in ["where", "filter", "only", "with discount",
                                 "greater than", "less than", "above", "below"]):
            filter_col = None
            filter_op = ">"
            filter_val = 0

            if "discount" in q:
                filter_col = "discount_pct"
                filter_op = ">"
                filter_val = 0
            elif "rating" in q:
                filter_col = "customer_rating"
                nums = re.findall(r'\d+\.?\d*', q)
                filter_val = float(nums[0]) if nums else 4.0
                filter_op = ">=" if "above" in q or "at least" in q else ">"

            if not filter_col:
                for c in mentioned_cols:
                    if c in numeric_cols:
                        filter_col = c
                        break

            if filter_col:
                nums = re.findall(r'\d+\.?\d*', q)
                if nums:
                    filter_val = float(nums[-1])
                plan.append({"tool": "filter_data",
                             "args": {"column": filter_col, "operator": filter_op,
                                      "value": filter_val}})

        # Intent: chart suggestion (always add if we have other tools in the plan)
        if plan or any(w in q for w in ["chart", "plot", "visualize", "graph", "show"]):
            x_col = None
            y_col = None
            for p in plan:
                if p["tool"] == "group_aggregate":
                    x_col = p["args"].get("group_col")
                    y_col = p["args"].get("value_col")
                elif p["tool"] == "compute_statistics":
                    y_col = p["args"].get("column")

            plan.append({"tool": "suggest_chart",
                         "args": {"question": question, "x_column": x_col,
                                  "y_column": y_col}})

        if not plan:
            plan.append({"tool": "describe_data", "args": {}})
            plan.append({"tool": "suggest_chart",
                         "args": {"question": question}})

        return plan

    def _format_answer(self, question: str, results: list[ToolResult]) -> str:
        """Format tool results into a human-readable answer."""
        parts = [f"**Question:** {question}\n"]

        for r in results:
            if not r.success:
                parts.append(f"**Error ({r.tool_name}):** {r.error}\n")
                continue

            if r.tool_name == "describe_data":
                d = r.data
                parts.append(f"**Data Overview:**\n")
                parts.append(f"- File: {d['file']} ({d['shape']['rows']} rows, {d['shape']['columns']} columns)\n")
                parts.append(f"- Numeric columns: {', '.join(d['columns']['numeric'])}\n")
                parts.append(f"- Categorical columns: {', '.join(d['columns']['categorical'])}\n")

            elif r.tool_name == "compute_statistics":
                d = r.data
                parts.append(f"\n**Statistics for `{d['column']}`:**\n")
                for k, v in d.items():
                    if k not in ("column", "type", "missing"):
                        parts.append(f"- {k}: {v:,.4f}\n" if isinstance(v, float) else f"- {k}: {v}\n")

            elif r.tool_name == "group_aggregate":
                d = r.data
                parts.append(f"\n**{d['agg_func'].title()} of `{d['value_col']}` by `{d['group_col']}`:**\n")
                sorted_results = sorted(d["results"].items(), key=lambda x: -(x[1] or 0))
                for k, v in sorted_results:
                    parts.append(f"- {k}: {v:,.2f}\n" if v else f"- {k}: N/A\n")

            elif r.tool_name == "filter_data":
                d = r.data
                parts.append(f"\n**Filter: `{d['column']}` {d['operator']} {d['value']}**\n")
                parts.append(f"- Matching rows: {d['matching_rows']:,} / {d['total_rows']:,} ({d['match_pct']}%)\n")
                if "numeric_summary" in d:
                    for col, stats in d["numeric_summary"].items():
                        parts.append(f"- {col}: mean={stats['mean']:,.2f}, sum={stats['sum']:,.2f}\n")

            elif r.tool_name == "suggest_chart":
                d = r.data
                parts.append(f"\n**Chart Recommendation:**\n")
                if d["primary"]:
                    p = d["primary"]
                    parts.append(f"- Type: **{p['chart_type']}**\n")
                    parts.append(f"- Reason: {p['reason']}\n")
                    if p.get("x"):
                        parts.append(f"- X-axis: `{p['x']}`\n")
                    if p.get("y"):
                        parts.append(f"- Y-axis: `{p['y']}`\n")
                if len(d["suggestions"]) > 1:
                    parts.append(f"- Alternative: {d['suggestions'][1]['chart_type']} — {d['suggestions'][1]['reason']}\n")

        return "".join(parts)

    def ask(self, question: str) -> dict:
        """Answer a question about the loaded data."""
        # Plan
        plan = self._detect_intent(question)
        if plan and "error" in plan[0]:
            return {"question": question, "answer": plan[0]["error"],
                    "tool_calls": [], "plan": plan}

        # Execute tools
        results = []
        for step in plan:
            result = self.registry.call(step["tool"], step["args"])
            results.append(result)

        # Format answer
        answer = self._format_answer(question, results)

        self.conversation.append({
            "question": question,
            "plan": plan,
            "results": [{"tool": r.tool_name, "success": r.success,
                          "time_ms": r.execution_time_ms} for r in results],
            "answer": answer,
        })

        return {
            "question": question,
            "answer": answer,
            "tool_calls": plan,
            "results": results,
        }


agent = DataAnalystAgent(registry)
print("DataAnalystAgent ready.")

## 8. Using the Agent — Load and Explore

In [ ]:
# Step 1: Load the data
load_result = registry.call("load_file", {"file_path": str(csv_path)})
print(f"Loaded: {load_result.data['file']}")
print(f"Rows: {load_result.data['rows']}")
print(f"Columns: {load_result.data['columns']}")

In [ ]:
# Ask: What's in this data?
response = agent.ask("Give me an overview of this data. What columns are available?")
print(response["answer"])

print("\n--- Tool Calls ---")
for tc in response["tool_calls"]:
    print(f"  {tc['tool']}({json.dumps({k: str(v)[:40] for k, v in tc['args'].items()})})")

## 9. Answering Statistics Questions

In [ ]:
response = agent.ask("What is the average revenue and what's the median?")
print(response["answer"])

In [ ]:
response = agent.ask("What is the total revenue by region?")
print(response["answer"])

In [ ]:
response = agent.ask("Show me the average customer rating by product")
print(response["answer"])

## 10. Filtering and Conditional Analysis

In [ ]:
response = agent.ask("How many orders have a discount greater than 10%? What's the revenue?")
print(response["answer"])

In [ ]:
response = agent.ask("Show me orders where customer rating is above 4.5")
print(response["answer"])

## 11. Chart Suggestions in Action

In [ ]:
questions_for_charts = [
    "What's the revenue trend over time by month?",
    "Compare total revenue by product",
    "What's the distribution of customer ratings?",
    "Is there a relationship between quantity and unit price?",
    "What's the revenue proportion by channel?",
    "Which are the top regions by total revenue?",
]

for q in questions_for_charts:
    response = agent.ask(q)
    chart = response["results"][-1]  # Last result is always suggest_chart
    if chart.success and chart.data["primary"]:
        p = chart.data["primary"]
        print(f"Q: {q}")
        print(f"   → {p['chart_type']} (x={p.get('x')}, y={p.get('y')})")
        print(f"   Reason: {p['reason']}")
        print()

## 12. Rendering the Suggested Charts

In [ ]:
def render_chart(suggestion: dict, data: pd.DataFrame):
    """Render a chart based on the agent's suggestion."""
    chart_type = suggestion["chart_type"]
    x = suggestion.get("x")
    y = suggestion.get("y")
    config = suggestion.get("config", {})

    fig, ax = plt.subplots(figsize=(10, 5))

    if chart_type == "bar":
        if x and y and x in data.columns and y in data.columns:
            grouped = data.groupby(x)[y].mean().sort_values(ascending=False)
            grouped.plot(kind="bar", ax=ax, color="#1f77b4", alpha=0.8)
            ax.set_ylabel(f"Mean {y}")
        ax.set_title(f"Mean {y} by {x}")

    elif chart_type == "horizontal_bar":
        if x and y and y in data.columns:
            agg_col = x if x in data.columns else y
            group_col = y if y in data.columns and not pd.api.types.is_numeric_dtype(data[y]) else x
            if group_col and agg_col:
                grouped = data.groupby(group_col)[agg_col].sum().sort_values()
                grouped.plot(kind="barh", ax=ax, color="#2ca02c", alpha=0.8)
                ax.set_xlabel(f"Total {agg_col}")
        ax.set_title("Ranked comparison")

    elif chart_type == "line":
        if "month" in data.columns and y and y in data.columns:
            monthly = data.groupby("month")[y].sum()
            monthly.plot(ax=ax, marker="o" if config.get("marker") else None)
            ax.set_ylabel(y)
            ax.tick_params(axis="x", rotation=45)
        ax.set_title(f"{y} over time")

    elif chart_type == "histogram":
        col = x or y
        if col and col in data.columns:
            data[col].hist(bins=config.get("bins", 20), ax=ax, alpha=0.7, color="#ff7f0e", edgecolor="black")
            if config.get("kde"):
                data[col].plot.kde(ax=ax.twinx(), color="red", linewidth=2)
            ax.set_xlabel(col)
        ax.set_title(f"Distribution of {col}")

    elif chart_type == "scatter":
        if x and y and x in data.columns and y in data.columns:
            ax.scatter(data[x], data[y], alpha=config.get("alpha", 0.5), s=20)
            ax.set_xlabel(x)
            ax.set_ylabel(y)
        ax.set_title(f"{y} vs {x}")

    elif chart_type in ("pie", "donut"):
        col = x or y
        if col and col in data.columns:
            counts = data[col].value_counts()
            if config.get("max_slices") and len(counts) > config["max_slices"]:
                top = counts.head(config["max_slices"] - 1)
                top["Other"] = counts.iloc[config["max_slices"] - 1:].sum()
                counts = top
            wedges, texts, autotexts = ax.pie(
                counts, labels=counts.index, autopct="%1.1f%%",
                startangle=90, pctdistance=0.85 if config.get("donut") else 0.5)
            if config.get("donut"):
                centre = plt.Circle((0, 0), 0.55, fc="white")
                ax.add_artist(centre)
        ax.set_title(f"Breakdown by {col}")

    plt.tight_layout()
    plt.show()


print("render_chart() defined — renders any chart suggestion from the agent.")

In [ ]:
# Render charts for a few questions
demo_questions = [
    "Compare total revenue by region",
    "What's the distribution of customer ratings?",
    "Revenue trend over time by month",
    "Is there a relationship between quantity and unit price?",
]

for q in demo_questions:
    response = agent.ask(q)
    last_result = response["results"][-1]
    if last_result.success and last_result.data.get("primary"):
        print(f"\nQ: {q}")
        render_chart(last_result.data["primary"], df)

## 13. Multi-Step Questions

Complex questions require multiple tool calls. The agent plans a sequence of operations and chains the results together.

```text
"What's the total revenue by region, and which region has the highest?"

Plan:
1. group_aggregate(group_col="region", value_col="revenue", agg_func="sum")
2. suggest_chart(question=..., x_column="region", y_column="revenue")

Answer assembles both results.
```


In [ ]:
multi_step_questions = [
    "What's the total revenue by region?",
    "What is the average revenue by product? Show me a chart.",
    "How many orders with discount greater than 15%? What's the breakdown?",
]

for q in multi_step_questions:
    response = agent.ask(q)
    print(f"Q: {q}")
    print(f"Tool calls: {' → '.join(tc['tool'] for tc in response['tool_calls'])}")
    print(f"All succeeded: {all(r.success for r in response['results'])}")
    print()

## 14. Excel (XLSX) Support

In [ ]:
# Reload from XLSX to demonstrate format flexibility
xlsx_result = registry.call("load_file", {"file_path": str(xlsx_path)})
print(f"Loaded XLSX: {xlsx_result.data['file']}")
print(f"Rows: {xlsx_result.data['rows']}, Columns: {len(xlsx_result.data['columns'])}")

# Same questions work identically
response = agent.ask("What is the average revenue by channel?")
print(f"\n{response['answer']}")

## 15. Tool-Call Observability

In [ ]:
print(f"Total tool calls this session: {len(registry.call_log)}")
print()

# Summary table
log_df = pd.DataFrame(registry.call_log)
print("Tool usage summary:")
print(log_df.groupby("tool").agg(
    calls=("tool", "count"),
    success_rate=("success", "mean"),
    avg_time_ms=("time_ms", "mean"),
).round(2).to_string())

print(f"\nLast 10 calls:")
for entry in registry.call_log[-10:]:
    status = "OK" if entry["success"] else f"FAIL: {entry['error']}"
    print(f"  {entry['tool']:25s} {entry['time_ms']:6.1f}ms  {status}")

## 16. Error Handling and Validation

Tool calling must be robust. Invalid column names, wrong types, missing data — the agent should handle these gracefully, not crash.


In [ ]:
# Test: invalid column name
result = registry.call("compute_statistics", {"column": "nonexistent_column"})
print(f"Invalid column: success={result.success}, error={result.error}")

# Test: missing required argument
result = registry.call("compute_statistics", {})
print(f"Missing arg:    success={result.success}, error={result.error}")

# Test: invalid operator
result = registry.call("filter_data", {"column": "revenue", "operator": "LIKE", "value": 100})
print(f"Invalid op:     success={result.success}, error={result.error}")

# Test: unsupported file format
result = registry.call("load_file", {"file_path": "data.json"})
print(f"Bad format:     success={result.success}, error={result.error}")

## 17. Production Integration — LLM Tool Calling

### OpenAI Function Calling Format

```python
tools = [
    {
        "type": "function",
        "function": {
            "name": "compute_statistics",
            "description": "Compute descriptive statistics for a column.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Column name"},
                    "stats": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Stats to compute"
                    },
                },
                "required": ["column"],
            },
        },
    },
    # ... other tools
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

# Handle tool calls
if response.choices[0].message.tool_calls:
    for tool_call in response.choices[0].message.tool_calls:
        name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        result = registry.call(name, args)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(result.data),
        })
```

### Anthropic Tool Use Format

```python
tools = [
    {
        "name": "compute_statistics",
        "description": "Compute descriptive statistics for a column.",
        "input_schema": {
            "type": "object",
            "properties": {
                "column": {"type": "string", "description": "Column name"},
            },
            "required": ["column"],
        },
    }
]
```

### Key Differences Between Providers

| Feature | OpenAI | Anthropic | Local (Ollama) |
|---|---|---|---|
| Format | `tools` array | `tools` array | `tools` array (same as OpenAI) |
| Multi-tool | Parallel calls | Sequential only | Sequential only |
| Streaming | Yes | Yes | Yes |
| Forced tool use | `tool_choice={"type":"function","function":{"name":"..."}}` | `tool_choice={"type":"tool","name":"..."}` | Not supported |

### The Agent Loop

```python
while True:
    response = llm.generate(messages, tools=tools)

    if response.has_tool_calls:
        for call in response.tool_calls:
            result = registry.call(call.name, call.args)
            messages.append(tool_result(call.id, result))
    else:
        print(response.text)
        break
```


## 18. Tool-Calling Design Patterns

### Pattern 1: Stateless Tools

Each tool call is independent. No shared state between calls.

```python
# Good: tool takes everything it needs
def compute_stats(data: pd.DataFrame, column: str) -> dict: ...

# Bad: tool relies on global state
def compute_stats(column: str) -> dict:
    return global_df[column].describe()  # Where did global_df come from?
```

**Our agent uses shared state** (`_agent_state["df"]`) for simplicity, but production systems should pass data explicitly or use session-scoped state.

### Pattern 2: Tool Chaining

The output of one tool feeds into the next:

```text
load_file("sales.csv")
    → columns: [revenue, region, ...]
        → group_aggregate("region", "revenue", "sum")
            → {North: 50K, South: 40K, ...}
                → suggest_chart(x="region", y="revenue")
                    → bar chart
```

### Pattern 3: Fallback Chains

If the primary tool fails, try alternatives:

```text
compute_statistics("revenue")
    → FAIL: column not found
        → describe_data()
            → columns are [Revenue, Region, ...]  (capital R!)
                → compute_statistics("Revenue")
                    → SUCCESS
```

### Pattern 4: Confirmation Before Action

For destructive operations, add a confirmation step:

```text
User: "Delete all rows with revenue < 100"
Agent: "This will remove 45 rows (9%). Confirm? [yes/no]"
User: "yes"
Agent: → delete_rows(filter="revenue < 100")
```

### Anti-Patterns to Avoid

| Anti-pattern | Problem | Fix |
|---|---|---|
| Too many small tools | LLM gets confused choosing | Combine related operations |
| Overlapping tool descriptions | LLM picks the wrong one | Make descriptions mutually exclusive |
| No error messages | LLM retries blindly | Return clear error messages with available options |
| Unbounded output | Token limit exceeded | Truncate or summarize large results |
| Missing validation | Injection risk, crashes | Validate all inputs against schema |


## 19. Full Conversation Flow

In [ ]:
# Reload CSV for clean state
registry.call("load_file", {"file_path": str(csv_path)})

conversation_questions = [
    "What data do I have?",
    "What's the average revenue?",
    "Show total revenue by region",
    "Which product has the highest average customer rating?",
    "How many orders have discounts above 10%?",
    "What's the revenue trend over time?",
]

print("FULL CONVERSATION")
print("=" * 70)
analysis_agent = DataAnalystAgent(registry)

for q in conversation_questions:
    response = analysis_agent.ask(q)
    tools_used = " → ".join(tc["tool"] for tc in response["tool_calls"])
    print(f"\n{'─' * 70}")
    print(f"USER: {q}")
    print(f"TOOLS: {tools_used}")
    print(response["answer"])

## 20. Agent Quality Evaluation

In [ ]:
# Define test cases with expected behaviors
test_cases = [
    {
        "question": "What's the average revenue?",
        "expected_tools": ["compute_statistics"],
        "expected_column": "revenue",
    },
    {
        "question": "Total revenue by region",
        "expected_tools": ["group_aggregate"],
        "expected_args": {"group_col": "region", "agg_func": "sum"},
    },
    {
        "question": "Overview of the data",
        "expected_tools": ["describe_data"],
    },
    {
        "question": "Distribution of customer ratings",
        "expected_tools": ["suggest_chart"],
        "expected_chart": "histogram",
    },
    {
        "question": "Compare revenue by product",
        "expected_tools": ["group_aggregate", "suggest_chart"],
    },
    {
        "question": "Orders with discount above 15%",
        "expected_tools": ["filter_data"],
    },
]

test_agent = DataAnalystAgent(registry)
results = []

for tc in test_cases:
    response = test_agent.ask(tc["question"])
    actual_tools = [c["tool"] for c in response["tool_calls"]]

    # Check tool selection
    expected = tc["expected_tools"]
    tool_match = all(t in actual_tools for t in expected)

    # Check all succeeded
    all_success = all(r.success for r in response["results"])

    results.append({
        "question": tc["question"][:40],
        "expected_tools": ", ".join(expected),
        "actual_tools": ", ".join(actual_tools),
        "tool_match": tool_match,
        "all_success": all_success,
    })

results_df = pd.DataFrame(results)
print("AGENT EVALUATION")
print("=" * 70)
print(results_df.to_string(index=False))
print(f"\nTool selection accuracy: {results_df['tool_match'].mean():.0%}")
print(f"Execution success rate: {results_df['all_success'].mean():.0%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Tool usage frequency
ax = axes[0]
all_tools_used = []
for entry in analysis_agent.conversation:
    for tc in entry["plan"]:
        all_tools_used.append(tc["tool"])
tool_freq = pd.Series(all_tools_used).value_counts()
tool_freq.plot(kind="barh", ax=ax, color="#1f77b4", alpha=0.8)
ax.set_xlabel("Times Called")
ax.set_title("Tool Usage Frequency")

# Success rate by tool
ax2 = axes[1]
log_df_full = pd.DataFrame(registry.call_log)
if not log_df_full.empty:
    success_by_tool = log_df_full.groupby("tool")["success"].mean()
    colors = ["#2ca02c" if v >= 0.9 else "#ff7f0e" if v >= 0.7 else "#d62728" for v in success_by_tool]
    success_by_tool.plot(kind="barh", ax=ax2, color=colors, alpha=0.8)
    ax2.set_xlabel("Success Rate")
    ax2.set_title("Tool Success Rate")
    ax2.set_xlim([0, 1.1])

plt.tight_layout()
plt.show()

## 21. Limitations

| Limitation | Impact | Production fix |
|---|---|---|
| **Rule-based intent** | Cannot handle novel question phrasings | Use LLM for intent detection and tool dispatch |
| **Single DataFrame** | Cannot join or merge multiple files | Add `join_files` tool |
| **No derived columns** | Cannot compute "revenue per unit" on the fly | Add `create_column` tool with expression parser |
| **Fixed chart types** | Limited to 6 chart types | Add heatmaps, box plots, violin plots, treemaps |
| **No memory across sessions** | Cannot reference previous analyses | Add conversation persistence |
| **No data cleaning** | Assumes clean data | Add `clean_column` tool (fill NA, fix types, dedup) |
| **Row-level results not returned** | Filter shows counts but not rows | Add `get_rows` tool with pagination |
| **English only** | Intent detection is English keyword-based | Use multilingual LLM for NLU |

### Security Considerations

| Risk | Mitigation |
|---|---|
| **Path traversal** in `load_file` | Restrict to allowed directories; validate path |
| **Arbitrary code execution** | No `eval()` or code generation; all operations are pre-built tools |
| **Output size** | Truncate results > N rows; summarize instead of returning raw data |
| **Resource exhaustion** | Set timeout on tool calls; limit file size and row count |


## 22. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "data_analyst_agent",
    "dataset": {
        "file": "sales_data.csv",
        "rows": len(df),
        "columns": list(df.columns),
    },
    "tools_registered": [t["name"] for t in registry.list_tools()],
    "total_tool_calls": len(registry.call_log),
    "success_rate": sum(1 for c in registry.call_log if c["success"]) / len(registry.call_log)
        if registry.call_log else 0,
    "conversation_turns": len(analysis_agent.conversation),
}

log_path = ARTIFACT_DIR / "data_analyst_agent_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")
print(f"Saved: {log_path}")

## 23. Key Takeaways

### What We Built
- A **data analyst agent** with 6 tools: file loading, describe, statistics, group/aggregate, filter, chart suggestion
- A **typed tool registry** with validation, dispatch, and logging
- **Multi-step reasoning**: agent chains tool calls to answer complex questions
- **Chart recommendation engine** that matches question intent to visualization type

### Tool-Calling Design Principles
1. **Tools should be simple** — one well-defined operation per tool
2. **Schemas are documentation** — the LLM reads them to decide which tool to use
3. **Validation before execution** — catch bad inputs before they cause errors
4. **Structured inputs and outputs** — dicts and typed parameters, not free text
5. **Log everything** — every tool call should be traceable for debugging

### Tool Design Checklist
- [ ] Clear, non-overlapping descriptions (LLM can distinguish between tools)
- [ ] All parameters are typed with descriptions
- [ ] Required vs optional is explicit
- [ ] Enum constraints on categorical parameters
- [ ] Error messages include available options (e.g., "Column not found. Available: [...]")
- [ ] Output is bounded (no returning 1M rows)
- [ ] Dangerous operations require confirmation

### When to Use This Pattern
| Use case | Good fit? | Why |
|---|---|---|
| Business dashboards | Yes | Non-technical users ask questions in plain English |
| Data exploration | Yes | Faster than writing pandas code |
| Automated reporting | Yes | Same questions run on new data each period |
| Ad-hoc SQL replacement | Partially | Works for simple queries; complex joins need more tools |
| Data science workflows | No | Scientists need code flexibility, not pre-built tools |
